In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv("../data/raw/online_retail_II.csv")

In [33]:
cleaned_data = data.copy()

In [ ]:
cleaned_data["InvoiceDate"] = pd.to_datetime(
    cleaned_data["InvoiceDate"]
)

In [ ]:
cleaned_data.info()

dtype('<M8[ns]')

#### Handling Missing Values

Before removing or replacing missing values, we first measure the amount of missing data in each column. We will make column-specific decisions based on the importance of each field to our analysis.

In [19]:
missing_summary = pd.DataFrame({
    # Check the number of missing values in each column
    "Missing Values": cleaned_data.isnull().sum(),

    # Calculate the percentage of missing values in each column
    "Percentage": (cleaned_data.isnull().sum() / len(cleaned_data)) * 100
})
missing_summary

,Missing Values,Percentage
Invoice,0,0.000000
StockCode,0,0.000000
Description,4382,0.410541
Quantity,0,0.000000
InvoiceDate,0,0.000000
Price,0,0.000000
Customer ID,243007,22.766873
Country,0,0.000000


Missing values were identified in `Description` and `Customer ID`.

Instead of removing all rows containing missing values, we retain these transactions because they may still be useful for overall sales and product analysis.

`Customer ID` will only be required for customer-level analysis. Therefore, transactions with missing Customer IDs will be excluded when performing customer-specific analysis rather than being removed from the entire dataset.

In [20]:
# Count missing values in the important columns
print("Missing Description:", cleaned_data["Description"].isna().sum())
print("Missing Customer ID:", cleaned_data["Customer ID"].isna().sum())

Missing Description: 4382
Missing Customer ID: 243007


In [25]:

#*If these are accidental duplicates, they could artificially increase our sales and quantity calculations.
#*A duplicate row means every column is exactly the same as another row. 

print("Duplicate Transcations:", cleaned_data.duplicated().sum())

Duplicate Transcations: 34335


In [ ]:
# Remove completely duplicate rows
print("Rows before removing duplicates:", len(cleaned_data))
cleaned_data.drop_duplicates(inplace=True)

# Check how many rows remain
print("Rows after removing duplicates:", len(cleaned_data))

Rows before removing duplicates: 1033036
Rows after removing duplicates: 1033036


In [ ]:
# CHECK INVALID / SUSPICIOUS TRANSACTIONS

# Negative quantities
negative_quantity = (cleaned_data["Quantity"] < 0).sum()

# Zero quantities
zero_quantity = (cleaned_data["Quantity"] == 0).sum()

# Zero prices
zero_price = (cleaned_data["Price"] == 0).sum()

# Negative prices
negative_price = (cleaned_data["Price"] < 0).sum()

print("Negative quantity records:", negative_quantity) #likely returns/cancellations,
print("Zero quantity records:", zero_quantity)
print("Zero price records:", zero_price) #need investigation because they contribute no revenue
print("Negative price records:", negative_price) #suspicious

Negative quantity records: 22496
Zero quantity records: 0
Zero price records: 6014
Negative price records: 5


In [43]:
# Check products with zero price
zero_price_data = cleaned_data[cleaned_data["Price"] == 0]
# Number of unique products with zero price
print(
    "Unique products with zero price:",
    zero_price_data["StockCode"].nunique()
)

# Most common zero-price products
print(
    zero_price_data["StockCode"].value_counts().head(10)
)
zero_price_data[
    ["StockCode", "Description", "Quantity", "Invoice", "Country"]
].head(5)

Unique products with zero price: 2971
StockCode
46000M    17
79321     17
22501     17
23084     16
21116     15
22423     15
46000S    14
84990     13
35965     13
20713     13
Name: count, dtype: int64


,StockCode,Description,Quantity,Invoice,Country
263,21733,85123a mixed,-96,489464,United Kingdom
283,71477,short,-240,489463,United Kingdom
284,85123A,21733 mixed,-192,489467,United Kingdom
470,21646,NaN,-50,489521,United Kingdom
3114,20683,NaN,-44,489655,United Kingdom


#### Handling Invalid and Non-Sales Transactions

Negative quantities were retained in the cleaned transaction dataset because
they may represent cancellations, returns, or inventory adjustments rather
than invalid data.

Negative prices were removed because they are not valid sales prices.

Zero-price transactions were also retained in the cleaned transaction dataset
because their business meaning was not assumed without investigation.

For sales analysis, a separate `sales_data` dataset was created containing
only transactions with positive quantities and positive prices. This ensures
that revenue and sales metrics are calculated using actual positive-value
sales transactions.

In [45]:
# Check zero-price records by quantity type

zero_price_data = cleaned_data[cleaned_data["Price"] == 0]

print("Zero-price records with positive quantity:",
      (zero_price_data["Quantity"] > 0).sum()) #Could be free/promotional or adjustment records

print("Zero-price records with negative quantity:",
      (zero_price_data["Quantity"] < 0).sum()) #Mostly looks like returns/adjustments

# Check negative-quantity records with cancellation-style invoices
negative_quantity_data = cleaned_data[
    cleaned_data["Quantity"] < 0
]

print(
    "Negative quantity records with cancellation invoice:",
    negative_quantity_data["Invoice"].astype(str).str.startswith("C").sum() #Strong evidence that many negative quantities are cancellations
)

Zero-price records with positive quantity: 2621
Zero-price records with negative quantity: 3393
Negative quantity records with cancellation invoice: 19103


In [46]:
# Remove records with negative prices
cleaned_data = cleaned_data[
    cleaned_data["Price"] >= 0
].copy()

print("Rows after removing negative prices:", len(cleaned_data))

Rows after removing negative prices: 1033031


In [ ]:
# Create a separate dataset for actual sales transactions
sales_data = cleaned_data[
    (cleaned_data["Quantity"] > 0) &
    (cleaned_data["Price"] > 0)
].copy()

print("Sales records:", len(sales_data)) #?This is the dataset we'll primarily use later for revenue, AOV, product sales, country sales, monthly sales, etc.

Sales records: 1007914


In [49]:
# Check that no invalid quantities remain
print("Negative quantities:",(sales_data["Quantity"] < 0).sum())
print("Zero quantities:", (sales_data["Quantity"] == 0).sum())

# Check that no invalid prices remain
print( "Negative prices:", (sales_data["Price"] < 0).sum())
print( "Zero prices:", (sales_data["Price"] == 0).sum())

Negative quantities: 0
Zero quantities: 0
Negative prices: 0
Zero prices: 0


In [51]:
# Save the cleaned transaction dataset
cleaned_data.to_csv(
    "../data/processed/cleaned_transactions.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
